# Part C4 (Bonus, +10 pts): Attention with OpenAI Triton

Implement one head of scaled dot-product attention with [Triton](https://triton-lang.org/),
the Python-embedded language for GPU kernels you saw in the Advanced CUDA and Triton lecture.
This is the same operation as C1/C2/C3, written the Triton way: you write `tl.load`/`tl.store`
on pointers and `tl.dot` for the matmuls, and the compiler handles the shared memory, the
barriers, and the tensor-core instructions you placed by hand in the CUDA versions.

This is a **bonus** question. It is graded on a rubric by the TA, not by the local grader.
Run it on a GPU runtime (Colab, Insomnia, or your own GPU box).

You will complete two things:
- `attn_kernel`: a `@triton.jit` kernel that computes attention tile by tile with an online softmax.
- `my_attention`: the Python launcher that sets up the grid and calls the kernel.

The notebook checks your result against a PyTorch reference and benchmarks it against
`torch.nn.functional.scaled_dot_product_attention`, which dispatches to a fused
FlashAttention backend.

In [ ]:
import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), "This notebook needs a CUDA GPU (run it on Colab/Insomnia with a GPU runtime)."
device = torch.device("cuda")
torch.manual_seed(0)

## Problem setup

One attention head. Query, key, and value tensors `Q`, `K`, `V` each have shape `(N, d)`.

- scores `S = (1 / sqrt(d)) * Q @ K.T`, shape `(N, N)`
- weights `P = softmax(S)` row-wise (safe softmax: subtract the row max before `exp`)
- output `O = P @ V`, shape `(N, d)`

We use `float32` here, which is the usual precision for attention kernels and what
`tl.dot` maps onto the tensor cores. The CUDA parts C1/C2/C3 use `float64` for an exact
checksum cross-check; this bonus is checked against a PyTorch reference instead, so the
precision does not need to match them. We pick `N = 2048`, `d = 64`.

In [ ]:
N, d = 2048, 64
scale = 1.0 / (d ** 0.5)

Q = torch.randn(N, d, device=device, dtype=torch.float32)
K = torch.randn(N, d, device=device, dtype=torch.float32)
V = torch.randn(N, d, device=device, dtype=torch.float32)

def reference_attention(Q, K, V, scale):
    S = (Q @ K.transpose(0, 1)) * scale
    P = torch.softmax(S, dim=1)
    return P @ V

O_ref = reference_attention(Q, K, V, scale)
print("reference O shape", tuple(O_ref.shape), "checksum", O_ref.sum().item())

## `attn_kernel`

Write a `@triton.jit` kernel that computes one head of scaled dot-product attention:
`O = softmax(scale * Q @ K.T) @ V`, with a numerically stable softmax. `Q`, `K`, `V`, `O`
are row-major `(N, BD)` fp32 tensors passed as pointers with their strides; `BD` is the head
dimension. The hard constraint is that you must not materialize the full `N x N` score
matrix: process the keys and values in tiles of `BN` and carry the softmax state across
tiles. How you map Triton programs to the work, and what you keep per program, is yours to
decide. `tl.dot` is available for the tile matmuls.


In [ ]:
@triton.jit
def attn_kernel(
    Q, K, V, O,
    N, scale,
    sqm, sqd, skn, skd, svn, svd, som, sod,
    BM: tl.constexpr, BN: tl.constexpr, BD: tl.constexpr,
):
    # One program per block of BM query rows. The Q tile stays in registers
    # while we stream over the keys/values in tiles of BN, carrying the online
    # softmax state (running max m_i, running denominator l_i, and the
    # rescaled output accumulator acc), so the N x N scores never exist.
    pid = tl.program_id(0)
    offs_m = pid * BM + tl.arange(0, BM)        # query rows of this program
    offs_d = tl.arange(0, BD)                    # head dimension
    offs_n = tl.arange(0, BN)                    # key offsets within a tile
    mask_m = offs_m < N

    q = tl.load(Q + offs_m[:, None] * sqm + offs_d[None, :] * sqd,
                mask=mask_m[:, None], other=0.0)                 # (BM, BD)

    m_i = tl.full((BM,), float("-inf"), dtype=tl.float32)
    l_i = tl.zeros((BM,), dtype=tl.float32)
    acc = tl.zeros((BM, BD), dtype=tl.float32)

    for start_n in range(0, N, BN):
        cols = start_n + offs_n
        mask_n = cols < N
        k = tl.load(K + cols[:, None] * skn + offs_d[None, :] * skd,
                    mask=mask_n[:, None], other=0.0)             # (BN, BD)
        v = tl.load(V + cols[:, None] * svn + offs_d[None, :] * svd,
                    mask=mask_n[:, None], other=0.0)             # (BN, BD)

        s = tl.dot(q, tl.trans(k)) * scale                       # (BM, BN)
        s = tl.where(mask_n[None, :], s, float("-inf"))          # out-of-range keys -> weight 0

        m_new = tl.maximum(m_i, tl.max(s, axis=1))
        alpha = tl.exp(m_i - m_new)                              # rescale factor for old state
        p = tl.exp(s - m_new[:, None])
        l_i = l_i * alpha + tl.sum(p, axis=1)
        acc = acc * alpha[:, None] + tl.dot(p, v)
        m_i = m_new

    acc = acc / l_i[:, None]
    tl.store(O + offs_m[:, None] * som + offs_d[None, :] * sod, acc,
             mask=mask_m[:, None])


## `my_attention`

Write the launcher: allocate the `(N, d)` output, choose the grid and block sizes, and call
`attn_kernel` with the tensors' strides. Return the output tensor.


In [ ]:
def my_attention(Q, K, V, scale, BM=64, BN=64):
    N, d = Q.shape
    assert d == triton.next_power_of_2(d) and d >= 16, "BD must be a power of 2 >= 16 for tl.dot"
    O = torch.empty_like(Q)
    grid = (triton.cdiv(N, BM),)                 # one program per BM query rows
    attn_kernel[grid](
        Q, K, V, O,
        N, scale,
        Q.stride(0), Q.stride(1), K.stride(0), K.stride(1),
        V.stride(0), V.stride(1), O.stride(0), O.stride(1),
        BM=BM, BN=BN, BD=d,
    )
    return O

O_triton = my_attention(Q, K, V, scale)
err = (O_triton - O_ref).abs().max().item()
print("Triton checksum", O_triton.sum().item(), "| reference", O_ref.sum().item())
print("max abs error vs reference:", err)
assert err < 1e-2, f"Triton output does not match the reference (max abs err {err:.3g})"


## Benchmark against PyTorch SDPA

`torch.nn.functional.scaled_dot_product_attention` dispatches to a fused FlashAttention
backend. Compare your kernel's latency against it with `triton.testing.do_bench`. Which one
wins depends on the shapes and the GPU: at small sizes the fused path carries dispatch
overhead and a simple Triton kernel can match or beat it, while at large sizes the tuned
backend usually pulls ahead. The point is to see both in the same units and reason about why.

In [ ]:
from triton.testing import do_bench

def sdpa():
    # SDPA expects (batch, heads, seq, dim); add singleton batch and head dims.
    q = Q[None, None]; k = K[None, None]; v = V[None, None]
    return torch.nn.functional.scaled_dot_product_attention(q, k, v)

ms_triton = do_bench(lambda: my_attention(Q, K, V, scale))
ms_sdpa = do_bench(sdpa)
print(f"Triton attention: {ms_triton:.3f} ms")
print(f"PyTorch SDPA:     {ms_sdpa:.3f} ms")
faster, slower = ("SDPA", "Triton") if ms_sdpa < ms_triton else ("Triton", "SDPA")
print(f"{faster} is {max(ms_triton, ms_sdpa) / min(ms_triton, ms_sdpa):.1f}x faster than {slower} at this size")

# Required output format (handout): C4_execution_time in ms with 3 decimals.
print(f"{ms_triton:.3f}")


## What to submit

Submit this completed notebook (`c4_triton.ipynb`) with `attn_kernel` and `my_attention` filled in,
the correctness check passing, and the benchmark cell run. In your report, note the latency
of your Triton kernel versus PyTorch SDPA, say which was faster at this size, and explain why
(dispatch overhead at small sizes, versus the fused backend's tuned block sizes and
tensor-core use at large sizes).